# P1 Evaluation: IPN-Hand — Metrics and Keyframe Figures

**Independent copy:** original evaluation notebooks are preserved. This new notebook supports a fresh Kaggle Save & Run All from the fine-tuned checkpoint (no training), with verified native CUDA wheels selected from an attached Kaggle Dataset. Two separate qualitative protocols: isolated clip filmstrips and continuous GT/pred timelines.

**Important:** Python 3.13 needs correctly built cp313 wheels **plus matching Torch/CUDA/ABI/SM**. Attach the wheel builder output dataset containing `wheel_manifest.json`, both `.whl` files and `verify_install.py`. If the CUDA environment changed, the notebook fails before expensive inference rather than silently reporting a different backend.


**P1 standalone execution order:** run this notebook top-to-bottom; it excludes the old single-video debug pass, postprocessing ablation, and unrelated VideoMAE/computation comparisons. It only performs checkpoint inference + the requested isolated and continuous metrics/figures.


## 0. Runtime setup

This section prepares VideoMamba and its dependencies. The notebook first provides a PyTorch-reference fallback so model construction is robust. If the prebuilt CUDA wheels used by the previous notebook are available, Section 3 activates the native CUDA Mamba kernels before evaluation/latency benchmarking.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Core Python packages.
packages = [
    "setuptools<82",
    "wheel",
    "packaging",
    "ninja",
    "timm==0.4.12",
    "einops==0.7.0",
    "transformers==4.36.1",
    "opencv-python-headless",
    "matplotlib",
    "pandas",
    "numpy",
    "torchvision",
    "tqdm",
    "scikit-learn",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

VIDEOMAMBA_REPO = Path("/kaggle/working/VideoMamba")
P1_PINNED_SOURCE_COMMIT = "37355c26d0ae99ca2459f6d4044a5f509031a79f"
MAMBA_ROOT = VIDEOMAMBA_REPO / "mamba"
MAMBA_PACKAGE_DIR = MAMBA_ROOT / "mamba_ssm"
VIDEOMAMBA_PATH = VIDEOMAMBA_REPO / "videomamba" / "video_sm"

if not VIDEOMAMBA_REPO.exists():
    print("Cloning OpenGVLab/VideoMamba ...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/OpenGVLab/VideoMamba.git",
         str(VIDEOMAMBA_REPO)],
        check=True,
    )
else:
    print("VideoMamba already exists:", VIDEOMAMBA_REPO)

subprocess.run(["git","-C",str(VIDEOMAMBA_REPO),"fetch","--depth","1",
                "origin", P1_PINNED_SOURCE_COMMIT], check=True)
subprocess.run(["git","-C",str(VIDEOMAMBA_REPO),"checkout","--force",
                P1_PINNED_SOURCE_COMMIT],check=True)
assert subprocess.check_output(["git","-C",str(VIDEOMAMBA_REPO),
    "rev-parse","HEAD"], text=True).strip() == P1_PINNED_SOURCE_COMMIT

assert MAMBA_PACKAGE_DIR.is_dir(), MAMBA_PACKAGE_DIR
assert VIDEOMAMBA_PATH.is_dir(), VIDEOMAMBA_PATH

for p in [str(VIDEOMAMBA_PATH), str(MAMBA_ROOT)]:
    while p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

print("MAMBA_ROOT      :", MAMBA_ROOT)
print("VIDEOMAMBA_PATH:", VIDEOMAMBA_PATH)


In [ ]:
import types
import random
import importlib
import importlib.machinery

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from einops import rearrange, repeat
from torchvision import transforms
from IPython.display import display
from tqdm.auto import tqdm

# -------------------------------------------------------------------------
# Clean previous Mamba imports so this cell can be rerun.
# -------------------------------------------------------------------------
for name in list(sys.modules):
    if name == "mamba_ssm" or name.startswith("mamba_ssm."):
        del sys.modules[name]

for name in ["causal_conv1d", "causal_conv1d_cuda", "selective_scan_cuda"]:
    sys.modules.pop(name, None)

importlib.invalidate_caches()

# -------------------------------------------------------------------------
# Source-package namespace.
# -------------------------------------------------------------------------
mamba_pkg = types.ModuleType("mamba_ssm")
mamba_pkg.__file__ = str(MAMBA_PACKAGE_DIR / "__init__.py")
mamba_pkg.__path__ = [str(MAMBA_PACKAGE_DIR)]
mamba_pkg.__package__ = "mamba_ssm"

mamba_spec = importlib.machinery.ModuleSpec(
    name="mamba_ssm",
    loader=None,
    is_package=True,
)
mamba_spec.submodule_search_locations = [str(MAMBA_PACKAGE_DIR)]
mamba_pkg.__spec__ = mamba_spec
sys.modules["mamba_ssm"] = mamba_pkg

# -------------------------------------------------------------------------
# PyTorch fallback for causal_conv1d.
# -------------------------------------------------------------------------
causal_mod = types.ModuleType("causal_conv1d")

def causal_conv1d_fn(x, weight, bias=None, activation=None):
    seqlen = x.shape[-1]
    dim = x.shape[1]
    width = weight.shape[-1]

    if weight.ndim == 2:
        w = weight.unsqueeze(1)
    elif weight.ndim == 3:
        w = weight
    else:
        raise ValueError(f"Invalid causal_conv1d weight shape: {tuple(weight.shape)}")

    out = F.conv1d(
        x,
        w,
        bias=bias,
        padding=width - 1,
        groups=dim,
    )[..., :seqlen]

    if activation in ["silu", "swish"]:
        out = F.silu(out)
    return out

def causal_conv1d_update(*args, **kwargs):
    raise NotImplementedError(
        "causal_conv1d_update is not used in batch VideoMamba inference."
    )

causal_mod.causal_conv1d_fn = causal_conv1d_fn
causal_mod.causal_conv1d_update = causal_conv1d_update
sys.modules["causal_conv1d"] = causal_mod
sys.modules["causal_conv1d_cuda"] = types.ModuleType("causal_conv1d_cuda")
sys.modules["selective_scan_cuda"] = types.ModuleType("selective_scan_cuda")

# -------------------------------------------------------------------------
# PyTorch fallback for RMSNorm/Triton layernorm.
# -------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5, device=None, dtype=None):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.register_parameter("bias", None)

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def rms_norm_fn(
    x,
    weight,
    bias=None,
    residual=None,
    eps=1e-5,
    prenorm=False,
    residual_in_fp32=False,
):
    if residual is not None:
        if residual_in_fp32:
            residual = residual.float()
        x = x + residual
        res = x
    else:
        res = x

    normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps) * weight
    if bias is not None:
        normed = normed + bias

    if prenorm:
        return normed, res
    return normed

triton_ln = types.ModuleType("mamba_ssm.ops.triton.layernorm")
triton_ln.RMSNorm = RMSNorm
triton_ln.layer_norm_fn = None
triton_ln.rms_norm_fn = rms_norm_fn
sys.modules["mamba_ssm.ops.triton.layernorm"] = triton_ln

# Compatibility shim for some transformers versions.
try:
    import transformers.generation
    if not hasattr(transformers.generation, "GreedySearchDecoderOnlyOutput"):
        transformers.generation.GreedySearchDecoderOnlyOutput = object
        transformers.generation.SampleDecoderOnlyOutput = object
except Exception:
    pass

import mamba_ssm.ops.selective_scan_interface as ssi
import mamba_ssm.modules.mamba_simple as ms

def mamba_inner_fn_no_out_proj(
    xz,
    conv1d_weight,
    conv1d_bias,
    x_proj_weight,
    delta_proj_weight,
    A,
    B=None,
    C=None,
    D=None,
    delta_bias=None,
    B_proj_bias=None,
    C_proj_bias=None,
    delta_softplus=True,
):
    L = xz.shape[-1]
    delta_rank = delta_proj_weight.shape[1]
    d_state = A.shape[-1] * (1 if not A.is_complex() else 2)

    x, z = xz.chunk(2, dim=1)

    x = causal_conv1d_fn(
        x,
        rearrange(conv1d_weight, "d 1 w -> d w"),
        conv1d_bias,
        "silu",
    )

    x_dbl = F.linear(
        rearrange(x, "b d l -> (b l) d"),
        x_proj_weight,
    )

    delta = delta_proj_weight @ x_dbl[:, :delta_rank].t()
    delta = rearrange(delta, "d (b l) -> b d l", l=L)

    if B is None:
        B = x_dbl[:, delta_rank:delta_rank + d_state]
        if B_proj_bias is not None:
            B = B + B_proj_bias.to(dtype=B.dtype)
        if not A.is_complex():
            B = rearrange(B, "(b l) dstate -> b dstate l", l=L).contiguous()
        else:
            B = rearrange(
                B, "(b l) (dstate two) -> b dstate (l two)",
                l=L, two=2
            ).contiguous()

    if C is None:
        C = x_dbl[:, -d_state:]
        if C_proj_bias is not None:
            C = C + C_proj_bias.to(dtype=C.dtype)
        if not A.is_complex():
            C = rearrange(C, "(b l) dstate -> b dstate l", l=L).contiguous()
        else:
            C = rearrange(
                C, "(b l) (dstate two) -> b dstate (l two)",
                l=L, two=2
            ).contiguous()

    return ssi.selective_scan_ref(
        x,
        delta,
        A,
        B,
        C,
        D,
        z=z,
        delta_bias=delta_bias,
        delta_softplus=delta_softplus,
    )

# Reference implementation for robust model loading.
ssi.selective_scan_fn = ssi.selective_scan_ref
ssi.mamba_inner_fn = ssi.mamba_inner_ref
ssi.bimamba_inner_fn = ssi.bimamba_inner_ref
ssi.mamba_inner_fn_no_out_proj = mamba_inner_fn_no_out_proj

ms.selective_scan_fn = ssi.selective_scan_ref
ms.mamba_inner_fn = ssi.mamba_inner_ref
ms.bimamba_inner_fn = ssi.bimamba_inner_ref
ms.mamba_inner_fn_no_out_proj = mamba_inner_fn_no_out_proj

from models import videomamba_tiny

MAMBA_BACKEND = "pytorch_reference"

print("OpenCV         :", cv2.__version__)
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU            :", torch.cuda.get_device_name(0))
print("VideoMamba     : imported")
print("Mamba backend  :", MAMBA_BACKEND)


## 1. Global configuration

All experiment-changing parameters are centralized here. There is no hard-coded assumption that overlap must equal 50%.


In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------
# Kaggle inputs
# ---------------------------------------------------------------------
DATASET_ROOT = Path("/kaggle/input/datasets/soumicksarker/ipn-hand-dataset")
ANNOT_DIR = DATASET_ROOT / "annotations" / "annotations"
DATASET_VIDEOS_DIR = DATASET_ROOT / "videos" / "videos"

ANNOT_ALL_PATH = ANNOT_DIR / "Annot_List.txt"
ANNOT_TRAIN_PATH = ANNOT_DIR / "Annot_TrainList.txt"
ANNOT_TEST_PATH = ANNOT_DIR / "Annot_TestList.txt"

CHECKPOINT_PATH = Path(
    "/kaggle/input/models/hoanganh94/"
    "bestcheckpoint/pytorch/default/1/checkpoint-best.pth"
)

# P1 wheels are resolved in the next cell from the new Dataset manifest.
ACTIVATE_NATIVE_MAMBA_CUDA = False
CAUSAL_WHEEL = None
MAMBA_WHEEL = None

# ---------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------
NUM_FRAMES = 16
IMG_SIZE = 224
NUM_CLASSES = 14
LOAD_IPN_CHECKPOINT = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Strict FP32 policy.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.set_float32_matmul_precision("highest")
    except Exception:
        pass

# ---------------------------------------------------------------------
# Continuous recognition
# ---------------------------------------------------------------------
CONTINUOUS_VIDEO = "1CM1_2_R__224.avi"

WINDOW_SIZE = 64
WINDOW_STEP = 16              # 75% overlap
MODEL_FRAMES = NUM_FRAMES

ENABLE_SMOOTHING = False

# Baseline post-processing values from the previous notebook.
# Keep these while testing overlap as the only changed variable.
T_AVG_SEC = 2.0
MIN_GESTURE_SEC = 1.2
FILL_SAME_NEIGHBORS = True

# Evaluation tolerance only; it does NOT modify predictions.
THETA_SEC = 2.0

# Short-gesture-safe preset to test later as a separate experiment:
# T_AVG_SEC = 0.3
# MIN_GESTURE_SEC = 0.5
# FILL_SAME_NEIGHBORS = False

assert MODEL_FRAMES == 16
assert 0 < WINDOW_STEP <= WINDOW_SIZE
assert WINDOW_SIZE % MODEL_FRAMES == 0, (
    "WINDOW_SIZE must be divisible by MODEL_FRAMES for fixed-stride sampling."
)

OVERLAP_RATIO = 1.0 - WINDOW_STEP / WINDOW_SIZE

print("DEVICE          :", DEVICE)
print("Window size     :", WINDOW_SIZE)
print("Window step     :", WINDOW_STEP)
print("Window overlap  :", f"{OVERLAP_RATIO*100:.2f}%")
print("Model frames    :", MODEL_FRAMES)
print("T_AVG_SEC       :", T_AVG_SEC, "(post-processing)")
print("MIN_GESTURE_SEC :", MIN_GESTURE_SEC, "(post-processing)")
print("THETA_SEC       :", THETA_SEC, "(evaluation only)")


### P1 Wheel selection: fresh Kaggle cp313/cp312 runtime

The next cell reads the wheel builder's manifest, checks Python/torch/CUDA/ABI/SM and hashes, then binds only a compatible, validated pair. It fails if no compatible pair was attached.


In [ ]:
# P1 startup gate: select wheel pair using the builder's validated manifest.
# Attach the NEW Kaggle wheel dataset as an Input before running this notebook.
# This deliberately rejects cp312 binaries in a cp313 Python environment.
import sys, json, hashlib, re
from pathlib import Path
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
import torch

P1_EXPECTED_VIDEOMAMBA_COMMIT = "37355c26d0ae99ca2459f6d4044a5f509031a79f"
P1_WHEEL_DATASET_HINT = ""  # Optional substring of the mounted Kaggle dataset path.
P1_REQUIRE_NATIVE_CUDA = True  # Fail early instead of silently using slow reference inference.

def _p1_sha256(file):
    h = hashlib.sha256()
    with Path(file).open("rb") as f:
        for chunk in iter(lambda:f.read(8*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _p1_inspect_manifest(manifest_path):
    m = json.loads(manifest_path.read_text(encoding="utf-8"))
    e = m["environment"]
    python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    if e.get("python_tag") != python_tag:
        raise ValueError(f"Python ABI {e.get('python_tag')} != {python_tag}")
    if e.get("torch_version") != torch.__version__:
        raise ValueError(f"PyTorch {e.get('torch_version')} != {torch.__version__}")
    if str(e.get("torch_cuda")) != str(torch.version.cuda):
        raise ValueError(f"PyTorch CUDA {e.get('torch_cuda')} != {torch.version.cuda}")
    abi = int(bool(torch._C._GLIBCXX_USE_CXX11_ABI))
    if int(e.get("torch_cxx11_abi", -1)) != abi:
        raise ValueError("PyTorch C++11 ABI mismatch")
    expected_commit = P1_EXPECTED_VIDEOMAMBA_COMMIT
    if m.get("source", {}).get("commit") != expected_commit:
        raise ValueError("VideoMamba wheel source commit differs from pinned model source")
    if not m.get("verification", {}).get("cuda_smoke_test_passed", False):
        raise ValueError("Wheel builder CUDA smoke test did not pass")
    if not torch.cuda.is_available():
        raise ValueError("CUDA GPU is not enabled")
    actual_sms = {"".join(map(str,torch.cuda.get_device_capability(i)))
                  for i in range(torch.cuda.device_count())}
    if not actual_sms.issubset(set(map(str,e.get("target_sms",[])))):
        raise ValueError(f"GPU targets {actual_sms} not supported by {e.get('target_sms')}")
    compatible = set(sys_tags())
    selected = {}
    for package in ("causal_conv1d", "mamba_ssm"):
        item = m["wheels"][package]
        candidate = manifest_path.parent / item["file"]
        if not candidate.is_file():
            raise ValueError(f"Wheel file missing: {candidate}")
        if _p1_sha256(candidate) != item["sha256"]:
            raise ValueError(f"Wheel checksum mismatch: {candidate}")
        name, _, _, tags = parse_wheel_filename(candidate.name)
        if name.replace("-", "_") != package:
            raise ValueError(f"Wrong wheel package: {candidate.name}")
        if not (compatible & tags):
            raise ValueError(f"Incompatible Python/platform wheel tags: {candidate.name}")
        selected[package] = candidate
    return selected, m

assert torch.cuda.is_available(), "Enable a GPU accelerator in Kaggle first."
P1_WHEEL_CANDIDATES = sorted(Path("/kaggle/input").rglob("wheel_manifest.json"))
if P1_WHEEL_DATASET_HINT:
    P1_WHEEL_CANDIDATES = [p for p in P1_WHEEL_CANDIDATES
                           if P1_WHEEL_DATASET_HINT.lower() in str(p).lower()]
P1_MATCHES, P1_REJECTIONS = [], []
for manifest_path in P1_WHEEL_CANDIDATES:
    try:
        pair, manifest_data = _p1_inspect_manifest(manifest_path)
        P1_MATCHES.append((manifest_path, pair, manifest_data))
    except Exception as exc:
        P1_REJECTIONS.append((str(manifest_path), str(exc)))

print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| torch CUDA:", torch.version.cuda)
print("Found wheel manifests:", len(P1_WHEEL_CANDIDATES),
      "| matches:", len(P1_MATCHES))
for path, reason in P1_REJECTIONS[:8]:
    print("[REJECTED]", path, "because", reason)
if len(P1_MATCHES) != 1:
    raise RuntimeError(
        f"Expected exactly one ABI/torch/CUDA/SM-compatible verified wheel manifest; "
        f"found {len(P1_MATCHES)}. Attach the NEW wheel Dataset and optionally set "
        f"P1_WHEEL_DATASET_HINT to its path. Never rename/install cp312 as cp313.")
P1_WHEEL_MANIFEST, P1_WHEEL_PAIR, P1_WHEEL_METADATA = P1_MATCHES[0]
CAUSAL_WHEEL = P1_WHEEL_PAIR["causal_conv1d"]
MAMBA_WHEEL = P1_WHEEL_PAIR["mamba_ssm"]
ACTIVATE_NATIVE_MAMBA_CUDA = True
print("Selected wheel manifest:", P1_WHEEL_MANIFEST)
print("Causal wheel:", CAUSAL_WHEEL.name)
print("Mamba wheel :", MAMBA_WHEEL.name)


## 2. Dataset and annotations

This section is needed for isolated/continuous IPN-Hand evaluation. It is **not required** for architecture-only Params/GFLOPs benchmarking.


In [ ]:
import re

CLASS_CODES = [
    "D0X", "B0A", "B0B",
    "G01", "G02", "G03", "G04", "G05", "G06",
    "G07", "G08", "G09", "G10", "G11",
]

CLASS_DESCRIPTIONS = [
    "No gesture",
    "Pointing with one finger",
    "Pointing with two fingers",
    "Click with one finger",
    "Click with two fingers",
    "Throw up",
    "Throw down",
    "Throw left",
    "Throw right",
    "Open twice",
    "Double click with one finger",
    "Double click with two fingers",
    "Zoom in",
    "Zoom out",
]

CLASS_NAMES = [
    f"{code} - {desc}"
    for code, desc in zip(CLASS_CODES, CLASS_DESCRIPTIONS)
]

required_dataset_paths = {
    "video dir": DATASET_VIDEOS_DIR,
    "Annot_List": ANNOT_ALL_PATH,
    "Annot_TrainList": ANNOT_TRAIN_PATH,
    "Annot_TestList": ANNOT_TEST_PATH,
}

for name, p in required_dataset_paths.items():
    print(f"{name:18s}: {p} -> {p.exists()}")

assert DATASET_VIDEOS_DIR.is_dir(), (
    "Attach the IPN-Hand Kaggle dataset before running evaluation sections."
)
assert ANNOT_TEST_PATH.is_file(), ANNOT_TEST_PATH

def canonical_video_key(name):
    name = Path(str(name).strip()).stem.lower()
    return re.sub(r"[^a-z0-9]", "", name)

VIDEO_INDEX = {
    canonical_video_key(p.name): p
    for p in DATASET_VIDEOS_DIR.glob("*.avi")
}

def resolve_video_path(video_name):
    return VIDEO_INDEX.get(canonical_video_key(video_name), None)

def read_ipn_annotations(path):
    raw = pd.read_csv(
        path,
        sep=",",
        header=None,
        dtype=str,
        engine="python",
    )

    if raw.shape[1] < 5:
        raise ValueError(
            f"{path.name} has {raw.shape[1]} columns; at least 5 are required."
        )

    df = raw.iloc[:, :5].copy()
    df.columns = ["video", "label", "id", "t_start", "t_end"]

    for col in ["video", "label"]:
        df[col] = df[col].astype(str).str.strip()

    for col in ["id", "t_start", "t_end"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["id", "t_start", "t_end"]).copy()
    df["id"] = df["id"].astype(int)
    df["t_start"] = df["t_start"].astype(int)
    df["t_end"] = df["t_end"].astype(int)
    df = df[df["id"].between(1, NUM_CLASSES)].copy()

    df["video_path"] = df["video"].apply(resolve_video_path)
    df["video_found"] = df["video_path"].notna()
    df["class_idx"] = df["id"] - 1
    df["class_code"] = df["class_idx"].map(CLASS_CODES.__getitem__)

    return df.reset_index(drop=True)

df_all = read_ipn_annotations(ANNOT_ALL_PATH)
df_train = read_ipn_annotations(ANNOT_TRAIN_PATH)
df_test = read_ipn_annotations(ANNOT_TEST_PATH)

summary = pd.DataFrame({
    "split": ["all", "train", "test"],
    "annotations": [len(df_all), len(df_train), len(df_test)],
    "mapped": [
        int(df_all["video_found"].sum()),
        int(df_train["video_found"].sum()),
        int(df_test["video_found"].sum()),
    ],
})
display(summary)

unmatched = df_all.loc[~df_all["video_found"], "video"].drop_duplicates()
if len(unmatched):
    print("Unmatched example:", unmatched.head(10).tolist())

print("AVI files:", len(VIDEO_INDEX))


## 3. VideoMamba model and common preprocessing

The model is first buildable with the reference fallback. If the prebuilt CUDA wheels are attached and compatible, this section reloads VideoMamba with native CUDA Mamba kernels. Architecture complexity does not depend on checkpoint weights, but evaluation requires the IPN-Hand checkpoint.


In [ ]:
def load_videomamba_model(
    checkpoint_path=CHECKPOINT_PATH,
    num_classes=NUM_CLASSES,
    device=DEVICE,
    load_checkpoint=LOAD_IPN_CHECKPOINT,
):
    model_vm = videomamba_tiny(
        img_size=IMG_SIZE,
        num_frames=NUM_FRAMES,
        kernel_size=1,
        num_classes=num_classes,
    )

    checkpoint_loaded = False

    if load_checkpoint and Path(checkpoint_path).is_file():
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
        state_dict = checkpoint.get("model", checkpoint)

        clean_state_dict = {}
        for k, v in state_dict.items():
            name = k[7:] if k.startswith("module.") else k
            clean_state_dict[name] = v

        model_vm.load_state_dict(clean_state_dict, strict=True)
        checkpoint_loaded = True
    elif load_checkpoint:
        print(
            "[WARNING] IPN-Hand checkpoint not found. "
            "Architecture benchmark can still run, but evaluation cannot."
        )

    model_vm = model_vm.float().to(device).eval()
    return model_vm, checkpoint_loaded


# First build using the robust reference import.
model, CHECKPOINT_LOADED = load_videomamba_model()

# -------------------------------------------------------------------------
# Optional native CUDA activation.
# -------------------------------------------------------------------------
if (
    ACTIVATE_NATIVE_MAMBA_CUDA
    and torch.cuda.is_available()
    and CAUSAL_WHEEL.is_file()
    and MAMBA_WHEEL.is_file()
):
    print("Installing prebuilt native Mamba CUDA wheels ...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "--no-deps",
            str(CAUSAL_WHEEL),
            str(MAMBA_WHEEL),
        ],
        check=True,
    )

    for name in list(sys.modules):
        if (
            name == "mamba_ssm"
            or name.startswith("mamba_ssm.")
            or name == "causal_conv1d"
            or name.startswith("causal_conv1d.")
            or name == "causal_conv1d_cuda"
            or name == "selective_scan_cuda"
        ):
            sys.modules.pop(name, None)

    importlib.invalidate_caches()

    try:
        import causal_conv1d_cuda
        import selective_scan_cuda
        import mamba_ssm
        import mamba_ssm.ops.selective_scan_interface as ssi_native
        import mamba_ssm.modules.mamba_simple as ms_native

        ssi_native = importlib.reload(ssi_native)
        ms_native = importlib.reload(ms_native)

        if ssi_native.selective_scan_fn is ssi_native.selective_scan_ref:
            raise RuntimeError("Native selective_scan_fn still points to reference path.")

        import models.videomamba as vm_mod
        vm_mod.Mamba = ms_native.Mamba
        videomamba_tiny = vm_mod.videomamba_tiny

        model, CHECKPOINT_LOADED = load_videomamba_model()
        MAMBA_BACKEND = "native_cuda"

    except Exception as e:
        print("[WARNING] Native CUDA activation failed:", repr(e))
        print("Continuing with the already-built reference model.")
        MAMBA_BACKEND = "pytorch_reference"
else:
    print("Native CUDA wheels not activated; using reference backend.")

print("Checkpoint loaded:", CHECKPOINT_LOADED)
print("Mamba backend    :", MAMBA_BACKEND)
print("Model device     :", next(model.parameters()).device)
print("Model dtype      :", next(model.parameters()).dtype)


### P1 Native CUDA model readiness gate

A small forward pass confirms the selected wheel backend before re-running isolated and continuous evaluation. The SHA-256 checkpoint/runtime manifest is recorded.


In [ ]:
# P1 fail-fast CUDA and checkpoint verification before the costly full test split.
import torch, hashlib, json
from pathlib import Path

assert torch.cuda.is_available(), "GPU not enabled."
assert MAMBA_BACKEND == "native_cuda", (
    "Native Mamba backend did not activate. Check wheel manifest/runtime compatibility.")
P1_MODEL_CHECKPOINT = Path(CHECKPOINT_PATH)
assert P1_MODEL_CHECKPOINT.is_file(), "Fine-tuned checkpoint missing."
assert next(model.parameters()).device.type == "cuda", "Model is not using CUDA."
with torch.inference_mode():
    p1_test_logits = model(torch.zeros(
        (1, 3, NUM_FRAMES, IMG_SIZE, IMG_SIZE), device=DEVICE, dtype=torch.float32))
torch.cuda.synchronize()
assert p1_test_logits.shape == (1, NUM_CLASSES), (
    f"Unexpected inference logits shape: {p1_test_logits.shape}")
assert torch.isfinite(p1_test_logits).all(), "NaN/Inf in CUDA output."
P1_CHECKPOINT_SHA256 = _p1_sha256(P1_MODEL_CHECKPOINT)
P1_RUNTIME_AUDIT = {
    "python": sys.version.split()[0], "torch": torch.__version__,
    "torch_cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "native_backend": MAMBA_BACKEND,
    "wheel_manifest": str(P1_WHEEL_MANIFEST),
    "checkpoint": str(P1_MODEL_CHECKPOINT),
    "checkpoint_sha256": P1_CHECKPOINT_SHA256,
    "videomamba_source_commit": P1_EXPECTED_VIDEOMAMBA_COMMIT,
    "frames": NUM_FRAMES, "image_size": IMG_SIZE, "num_classes": NUM_CLASSES
}
P1_AUDIT_FILE = Path("/kaggle/working/p1_cuda_runtime_audit.json")
P1_AUDIT_FILE.write_text(json.dumps(P1_RUNTIME_AUDIT, indent=2), encoding="utf-8")
print("[PASS] Native CUDA model forward smoke test; runtime audit:", P1_AUDIT_FILE)


In [ ]:
# ImageNet normalization used by the existing checkpoint pipeline.
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

def sample_indices(total_frames, num_frames=NUM_FRAMES):
    # Uniformly sample exactly num_frames over an isolated clip.
    if total_frames <= 0:
        return np.zeros(num_frames, dtype=np.int64)
    return np.linspace(
        0,
        total_frames - 1,
        num_frames,
    ).round().astype(np.int64)

def extract_frames_from_video(
    video_path,
    t_start=None,
    t_end=None,
    num_frames=NUM_FRAMES,
):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    total_vid_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # IPN-Hand annotations are 1-based.
    start_f = max(1, int(t_start)) if t_start is not None else 1
    end_f = (
        min(total_vid_frames, int(t_end))
        if t_end is not None
        else total_vid_frames
    )

    if end_f < start_f:
        cap.release()
        raise ValueError(f"Invalid frame range: {start_f} -> {end_f}")

    clip_len = end_f - start_f + 1
    sampled_rel_idx = sample_indices(clip_len, num_frames)
    sampled_abs_idx = start_f - 1 + sampled_rel_idx

    sampled_set = set(sampled_abs_idx.tolist())
    frames_dict = {}

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_f - 1)
    curr_idx = start_f - 1

    while curr_idx <= end_f - 1 and len(frames_dict) < len(sampled_set):
        ok, frame = cap.read()
        if not ok:
            break
        if curr_idx in sampled_set:
            frames_dict[curr_idx] = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        curr_idx += 1

    cap.release()

    frames = []
    last_valid = None
    for idx in sampled_abs_idx:
        idx = int(idx)
        if idx in frames_dict:
            frame = frames_dict[idx]
            last_valid = frame
        elif last_valid is not None:
            frame = last_valid
        else:
            frame = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        frames.append(frame)

    return np.asarray(frames), sampled_abs_idx + 1

def frames_to_tensor(frames_np, device=DEVICE):
    tensor = torch.stack([transform(frame) for frame in frames_np])
    tensor = tensor.permute(1, 0, 2, 3).unsqueeze(0).contiguous()
    return tensor.to(device=device, dtype=torch.float32, non_blocking=True)

def run_inference(model, frames_np, device=DEVICE):
    tensor = frames_to_tensor(frames_np, device)
    with torch.inference_mode():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1).squeeze(0)

    probs = probs.detach().cpu().numpy()
    pred_idx = int(np.argmax(probs))
    confidence = float(probs[pred_idx])
    return pred_idx, confidence, probs


## 4. Isolated gesture classification

Each annotated gesture interval is treated as one isolated clip. Exactly 16 frames are uniformly sampled across the annotated interval. No sliding window or post-processing is used.


In [ ]:
ISOLATED_INCLUDE_NO_GESTURE = False
ISOLATED_MAX_SAMPLES = None   # e.g. 100 for a quick test
ISOLATED_SAVE_CSV = True

assert CHECKPOINT_LOADED, "IPN-Hand checkpoint is required for evaluation."
assert "df_test" in globals(), "Run Section 2 first."

isolated_df = df_test[df_test["video_found"]].copy()
if not ISOLATED_INCLUDE_NO_GESTURE:
    isolated_df = isolated_df[isolated_df["class_idx"] != 0].copy()

if ISOLATED_MAX_SAMPLES is not None:
    isolated_df = isolated_df.iloc[:int(ISOLATED_MAX_SAMPLES)].copy()

records = []
t0 = __import__("time").perf_counter()

for row in tqdm(
    isolated_df.itertuples(index=False),
    total=len(isolated_df),
    desc="Isolated evaluation",
):
    frames_np, sampled_ids = extract_frames_from_video(
        row.video_path,
        row.t_start,
        row.t_end,
        NUM_FRAMES,
    )

    pred_idx, conf, probs = run_inference(model, frames_np, DEVICE)

    records.append({
        "video": row.video,
        "video_path": str(row.video_path),
        "t_start": int(row.t_start),
        "t_end": int(row.t_end),
        "duration_frames": int(row.t_end - row.t_start + 1),
        "gt_idx": int(row.class_idx),
        "gt_code": CLASS_CODES[int(row.class_idx)],
        "pred_idx": pred_idx,
        "pred_code": CLASS_CODES[pred_idx],
        "confidence": conf,
        "correct": pred_idx == int(row.class_idx),
        **{f"prob_{j}": float(p) for j, p in enumerate(probs)},
        "sampled_frame_indices": ",".join(map(str, sampled_ids.tolist())),
    })

isolated_results = pd.DataFrame(records)
isolated_elapsed = __import__("time").perf_counter() - t0

isolated_top1 = float(isolated_results["correct"].mean()) if len(isolated_results) else np.nan

per_class_isolated = (
    isolated_results
    .groupby(["gt_idx", "gt_code"], as_index=False)
    .agg(
        samples=("correct", "size"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean"),
    )
)

print(f"Samples       : {len(isolated_results)}")
print(f"Top-1 accuracy: {isolated_top1:.4f}")
print(f"Elapsed       : {isolated_elapsed:.1f} s")
display(per_class_isolated)

if ISOLATED_SAVE_CSV:
    ISOLATED_CSV = "/kaggle/working/videomamba_ipn_isolated_test.csv"
    isolated_results.to_csv(ISOLATED_CSV, index=False)
    print("Saved:", ISOLATED_CSV)


### 4.1 P1: Early isolated gesture recognition (GT-prefix)

Accuracy@10/20/50/75/100% and normalized observation-ratio AUC. This is **not** causal RSC Early@10. Reuses the full-length inference at 100%, and saves a CSV per prefix.


In [ ]:
# P1 isolated early recognition: first 10/20/50/75% of each annotated gesture.
# Unlike RSC Early@10, this is GT-boundary, isolated-prefix recognition.
# Runs additional GPU forward passes; the 100% baseline is reused from Section 4.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

P1_IPN_DIR = Path("/kaggle/working/p1_ipn_re_evaluation")
P1_IPN_DIR.mkdir(parents=True, exist_ok=True)
P1_PREFIX_FRACTIONS = (0.10, 0.20, 0.50, 0.75, 1.0)
RUN_P1_PREFIX = True
assert len(isolated_df) == len(isolated_results), "Partial/mismatched isolated evaluation."
assert CHECKPOINT_LOADED, "Load the original fine-tuned checkpoint first."

if RUN_P1_PREFIX:
    p1_early_rows = []
    p1_early_summary = []
    for frac in P1_PREFIX_FRACTIONS:
        fraction_rows = []
        if frac == 1.0:
            df_prefix = isolated_results.copy()
            df_prefix["observed_fraction"] = frac
            df_prefix["prefix_end"] = df_prefix["t_end"].astype(int)
        else:
            for row in tqdm(isolated_df.itertuples(index=False),
                            total=len(isolated_df), desc=f"IPN prefix {frac:.0%}"):
                start, end = int(row.t_start), int(row.t_end)
                prefix_end = min(end, start + max(1, int(np.ceil((end-start+1)*frac))) - 1)
                frames, indices = extract_frames_from_video(
                    row.video_path, start, prefix_end, NUM_FRAMES)
                pred, confidence, probs = run_inference(model, frames, DEVICE)
                fraction_rows.append({
                    "video": str(row.video), "video_path": str(row.video_path),
                    "t_start": start, "t_end": end, "prefix_end": prefix_end,
                    "observed_fraction": frac, "gt_idx": int(row.class_idx),
                    "pred_idx": int(pred), "confidence": float(confidence),
                    "correct": int(pred) == int(row.class_idx),
                    "sampled_frame_indices": ",".join(map(str, indices.tolist())),
                    **{f"prob_{j}": float(p) for j, p in enumerate(probs)},
                })
            df_prefix = pd.DataFrame(fraction_rows)
        assert len(df_prefix) == len(isolated_results), "Prefix count differs from baseline."
        df_prefix.to_csv(P1_IPN_DIR / f"isolated_prefix_{int(frac*100):03d}.csv", index=False)
        y = df_prefix["gt_idx"].to_numpy(dtype=int)
        yp = df_prefix["pred_idx"].to_numpy(dtype=int)
        p1_early_summary.append({
            "observation_ratio": frac, "samples": len(y),
            "accuracy": float(accuracy_score(y, yp)),
            "macro_f1": float(f1_score(
                y, yp, labels=list(range(1, NUM_CLASSES)),
                average="macro", zero_division=0)),
        })
    p1_early_table = pd.DataFrame(p1_early_summary).sort_values("observation_ratio")
    p1_early_table.to_csv(P1_IPN_DIR / "early_accuracy_curve.csv", index=False)
    x = p1_early_table["observation_ratio"].to_numpy()
    y = p1_early_table["accuracy"].to_numpy()
    auc = float(np.sum((y[1:] + y[:-1]) * np.diff(x) / 2) / (x[-1] - x[0]))
    (P1_IPN_DIR / "early_auc.json").write_text(json.dumps({
        "auc_accuracy_10_to_100": auc,
        "note": "Trapezoidal AUC normalized over observation ratios [0.10, 1.00]."
    }, indent=2))
    print(f"IPN normalized early accuracy AUC [10%,100%] = {auc:.5f}")
    display(p1_early_table)


### Additional isolated diagnostics: Top-2/3, balanced accuracy, class F1 and first/stable correct prefix

Uses existing isolated and prefix CSV predictions. Offline GT-defined observation ratio is NOT Early@10 online response delay.


In [ ]:
# P1 additional ISOLATED diagnostics: Top-2/3, balanced accuracy, per-class
# accuracy/precision/recall/F1 and earliest/stable correct observed GT-prefix ratio.
# Does not claim online latency; uses SAME already-saved prefix predictions.
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.metrics import (balanced_accuracy_score, precision_recall_fscore_support)
P1_EXTRA_ISOLATED_DIR = P1_IPN_DIR / "extra_isolated_metrics"
P1_EXTRA_ISOLATED_DIR.mkdir(parents=True,exist_ok=True)
P1_EXTRA_PREFIXES = [10,20,50,75,100]
P1_EXTRA_KEY = ["video","t_start","t_end"]
prefix_predictions=[]
summary_rows=[]
for percentage in P1_EXTRA_PREFIXES:
    file=P1_IPN_DIR / f"isolated_prefix_{percentage:03d}.csv"
    if not file.is_file():
        raise FileNotFoundError(f"Run P1 early-prefix cell first: {file}")
    table=pd.read_csv(file)
    prob_columns=[f"prob_{k}" for k in range(NUM_CLASSES)]
    if not set(prob_columns).issubset(table.columns):
        raise RuntimeError(f"Missing full class probability vector in {file}")
    if table.duplicated(P1_EXTRA_KEY).any():
        raise RuntimeError(f"Duplicate sample keys in {file}")
    prob=table[prob_columns].to_numpy(dtype=float)
    assert np.all(np.isfinite(prob)) and np.allclose(prob.sum(axis=1),1.,atol=1e-4)
    truth=table["gt_idx"].to_numpy(dtype=int)
    pred=table["pred_idx"].to_numpy(dtype=int)
    top2=np.argsort(-prob,axis=1)[:,:2]
    top3=np.argsort(-prob,axis=1)[:,:3]
    summary_rows.append({
        "observed_fraction":percentage/100,"samples":len(table),
        "top1_accuracy":float(np.mean(pred==truth)),
        "top2_accuracy":float(np.mean((top2==truth[:,None]).any(axis=1))),
        "top3_accuracy":float(np.mean((top3==truth[:,None]).any(axis=1))),
        "balanced_accuracy_present_classes":float(balanced_accuracy_score(truth,pred)),
        "high_confidence_wrong_ge_090_count":int(
            np.sum((pred!=truth)&(np.max(prob,axis=1)>=.90)))
    })
    table=table[P1_EXTRA_KEY+["gt_idx","pred_idx"]].copy()
    table["observed_fraction"]=percentage/100
    prefix_predictions.append(table)

summary=pd.DataFrame(summary_rows)
summary.to_csv(P1_EXTRA_ISOLATED_DIR/"topk_and_balanced_accuracy_by_ratio.csv",index=False)
# Use 100% evaluation as canonical sample identity, align by actual keys.
base=prefix_predictions[-1][P1_EXTRA_KEY+["gt_idx"]].copy()
matrix=[]
for table in prefix_predictions:
    compare=base.merge(table,on=P1_EXTRA_KEY,how="left",validate="one_to_one",
                       suffixes=("_base",""))
    if compare["pred_idx"].isna().any() or len(compare)!=len(base):
        raise RuntimeError("Missing prefix predictions in aligned test samples")
    assert np.array_equal(compare["gt_idx_base"], compare["gt_idx"]), "Ground truth mismatch"
    matrix.append((compare["gt_idx"].to_numpy()==compare["pred_idx"].to_numpy()))
correct=np.column_stack(matrix)
ratios=np.asarray(P1_EXTRA_PREFIXES,dtype=float)/100.
first=np.array([ratios[np.flatnonzero(r)[0]] if np.any(r) else np.nan for r in correct])
stable=np.array([next((float(ratios[j]) for j in range(len(ratios))
                      if np.all(r[j:])),np.nan) for r in correct])
samples=base.copy()
for j,r in enumerate(ratios):
    samples[f"correct_at_{int(r*100)}"]=correct[:,j].astype(int)
samples["first_correct_ratio"]=first
samples["first_stably_correct_ratio"]=stable
samples.to_csv(P1_EXTRA_ISOLATED_DIR/"early_correctness_per_sample.csv",index=False)
# Metrics explicitly distinguish conditional coverage among samples recognized
# at 100% from samples never recognized on the tested observation grid.
early_json={
    "samples":int(len(base)),
    "prefix_grid":list(map(float,ratios)),
    "ever_correct_fraction":float(np.mean(correct.any(axis=1))),
    "correct_at_100_fraction":float(np.mean(correct[:,-1])),
    "never_correct_count":int(np.sum(~correct.any(axis=1))),
    "early_correct_but_wrong_at_100_count":int(np.sum(correct[:,:-1].any(axis=1)&(~correct[:,-1]))),
    "mean_first_correct_ratio_conditional":float(np.nanmean(first)) if np.any(np.isfinite(first)) else None,
    "mean_stable_correct_ratio_conditional":float(np.nanmean(stable)) if np.any(np.isfinite(stable)) else None,
    "warning":"Offline GT-defined isolated prefixes only; not real-time prediction latency."
}
(P1_EXTRA_ISOLATED_DIR/"early_correctness_summary.json").write_text(
    json.dumps(early_json,indent=2))
y=isolated_results["gt_idx"].to_numpy(dtype=int)
yp=isolated_results["pred_idx"].to_numpy(dtype=int)
classes=np.arange(NUM_CLASSES)
prec,rec,f1,support=precision_recall_fscore_support(
    y,yp,labels=classes,zero_division=0)
per_class=pd.DataFrame({
    "class_idx":classes,"class_name":[CLASS_CODES[i] for i in classes],
    "support":support,"precision":prec,"recall":rec,"f1":f1
})
per_class.to_csv(P1_EXTRA_ISOLATED_DIR/"isolated_per_class_metrics.csv",index=False)
print("Top-K by prefix:");display(summary)
print("Early correctness (GT-prefix):",early_json)


### P1 Isolated full-export: EXACT sixteen model-input frames per clip

Export 100% of evaluated correct/error isolated clips into separate JPEG galleries (4×4 frames). Uses original model temporal sample_indices and spatial Resize/CenterCrop. Also saves complete sample metadata CSV and decode failures.


In [ ]:
# P1 ISOLATED: export ALL correct/error samples using the EXACT 16 temporal indices
# used by the checkpoint inference, with the same spatial Resize/CenterCrop.
# No model inference is repeated here. Output JPEG mosaics + complete CSV audit.
from pathlib import Path
import cv2, numpy as np, pandas as pd
from torchvision import transforms
from IPython.display import Image as IPyImage, display

P1_IS_IPN = True
P1_VIS_DIR = P1_IPN_DIR / "isolated_16frame"
P1_ISO_EXPORT_ALL = True
P1_ISO_LIMIT = None                  # None = 100%; optional debug cap
P1_ISO_TILE_SIZE = 192             # display size; model crop is 224x224
P1_ISO_JPEG_QUALITY = 87
P1_ISO_PREVIEW = 2
P1_VIS_DIR.mkdir(parents=True, exist_ok=True)
for subfolder in ("correct", "error"):
    (P1_VIS_DIR / subfolder).mkdir(exist_ok=True)

p1_vis_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
])
p1_iso_results = isolated_results.copy().reset_index(drop=True)
if not P1_IS_IPN:
    p1_iso_results = p1_iso_results.merge(
        test_segments[["sample_id", "video_path"]].drop_duplicates("sample_id"),
        on="sample_id", how="left", validate="one_to_one",
    )
assert p1_iso_results["video_path"].notna().all(), "Missing isolated video paths"
assert len(p1_iso_results) == len(isolated_results)
assert NUM_FRAMES == 16, "This figure protocol expects the checkpoint's 16 input frames"

def p1_mosaic_16(video_path, indices_zero_based, title, subtitle, output):
    path = Path(str(video_path))
    if not path.is_file():
        raise FileNotFoundError(path)
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open {path}")
    tiles = []
    try:
        for idx in indices_zero_based:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, bgr = cap.read()
            if not ok:
                raise RuntimeError(f"Cannot read model input frame {idx} of {path}")
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            # Exact model spatial view BEFORE normalization, resized only for the figure.
            rgb_crop = np.asarray(p1_vis_transform(rgb))
            tile = cv2.resize(
                cv2.cvtColor(rgb_crop, cv2.COLOR_RGB2BGR),
                (P1_ISO_TILE_SIZE, P1_ISO_TILE_SIZE),
                interpolation=cv2.INTER_AREA,
            )
            cv2.rectangle(tile, (0,0), (85,22), (0,0,0), -1)
            cv2.putText(tile, f"F{len(tiles)+1:02} #{int(idx)}", (4,16),
                        cv2.FONT_HERSHEY_SIMPLEX, .43, (255,255,255), 1, cv2.LINE_AA)
            tiles.append(tile)
    finally:
        cap.release()
    assert len(tiles) == 16
    body = np.vstack([np.hstack(tiles[i:i+4]) for i in (0,4,8,12)])
    hdr = np.full((78, body.shape[1], 3), 245, dtype=np.uint8)
    cv2.putText(hdr, str(title)[:98], (12,29), cv2.FONT_HERSHEY_SIMPLEX,
                .62, (20,20,20), 2, cv2.LINE_AA)
    cv2.putText(hdr, str(subtitle)[:115], (12,57), cv2.FONT_HERSHEY_SIMPLEX,
                .47, (40,40,40), 1, cv2.LINE_AA)
    image = np.vstack((hdr,body))
    if not cv2.imwrite(str(output), image, [cv2.IMWRITE_JPEG_QUALITY,
                                            P1_ISO_JPEG_QUALITY]):
        raise RuntimeError(f"Cannot save image {output}")

p1_iso_rows, p1_iso_failures = [], []
selected = p1_iso_results if P1_ISO_LIMIT is None else p1_iso_results.head(int(P1_ISO_LIMIT))
for number, case in enumerate(selected.itertuples(index=False), start=1):
    row = case._asdict()
    start, end = int(row["t_start"]), int(row["t_end"])
    source_indices = sample_indices(end-start+1, NUM_FRAMES) + start
    indices0 = source_indices - (1 if P1_IS_IPN else 0)
    gt, pred = int(row["gt_idx"]), int(row["pred_idx"])
    correct = gt == pred
    category = "correct" if correct else "error"
    output_file = P1_VIS_DIR / category / f"{category}_{number:05d}.jpg"
    try:
        p1_mosaic_16(
            row["video_path"], indices0,
            f"{category.upper()} | GT {CLASS_CODES[gt]} | PRED {CLASS_CODES[pred]}",
            f"sample {number} | confidence={float(row['confidence']):.4f} | "
            f"original frames {start}..{end} | source {Path(str(row['video_path'])).name}",
            output_file,
        )
        p1_iso_rows.append({
            "sample_number":number, "sample_id":str(row.get("sample_id", row.get("video",""))),
            "video_path":str(row["video_path"]), "original_start":start, "original_end":end,
            "gt_idx":gt, "pred_idx":pred, "correct":correct, "confidence":float(row["confidence"]),
            "model_input_indices_zero_based":",".join(map(str,indices0)),
            "image_path":str(output_file), "num_input_frames":16,
            "protocol":"GT-bounded isolated, original 16 temporal samples and Resize+CenterCrop",
        })
    except Exception as exc:
        p1_iso_failures.append({"sample_number":number, "video_path":str(row["video_path"]),
                                "error":str(exc)})
p1_iso_export = pd.DataFrame(p1_iso_rows)
p1_iso_export.to_csv(P1_VIS_DIR / "isolated_all_samples_index.csv", index=False)
pd.DataFrame(p1_iso_failures).to_csv(P1_VIS_DIR / "isolated_decode_failures.csv", index=False)
print(f"ISOLATED: exported {len(p1_iso_rows)}/{len(selected)} samples; "
      f"errors={sum(not x['correct'] for x in p1_iso_rows)}, "
      f"correct={sum(x['correct'] for x in p1_iso_rows)}, "
      f"decode failures={len(p1_iso_failures)}")
print("Output:",P1_VIS_DIR)
for group in ("error","correct"):
    for path in sorted((P1_VIS_DIR/group).glob("*.jpg"))[:P1_ISO_PREVIEW]:
        display(IPyImage(filename=str(path),width=680))


## 5. Continuous gesture recognition

Continuous recognition uses a sliding window over the full video. Window overlap is controlled only by `WINDOW_STEP`; there is no 50%-overlap assertion.

The processing order is:

`window inference → optional smoothing → window-to-frame probability aggregation → linking → short-segment removal → evaluation`


In [ ]:
import time

def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    cap.release()
    if fps <= 0:
        fps = 30.0
    return total_frames, fps

def build_window_starts(total_frames, window_size, window_step):
    if total_frames <= window_size:
        return [0]

    last_full_start = total_frames - window_size
    starts = list(range(0, last_full_start + 1, window_step))

    # Tail-align one full final window when stride does not land exactly at the end.
    if starts[-1] != last_full_start:
        starts.append(last_full_start)

    return starts

def build_sample_offsets(window_size, model_frames):
    assert window_size % model_frames == 0
    stride = window_size // model_frames
    offsets = np.arange(0, window_size, stride, dtype=np.int64)
    assert len(offsets) == model_frames
    return offsets

def read_sampled_window(
    video_path,
    start_frame,
    total_frames,
    sample_offsets,
):
    idxs = np.clip(
        int(start_frame) + sample_offsets,
        0,
        total_frames - 1,
    ).astype(np.int64)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    frames = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            raise RuntimeError(f"Cannot read frame {idx} from {video_path}")
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    cap.release()
    return np.asarray(frames), idxs

def smooth_window_probs(window_probs, enabled=False):
    if not enabled or len(window_probs) <= 1:
        return np.asarray(window_probs, dtype=np.float32).copy()

    padded = np.pad(
        window_probs,
        ((1, 1), (0, 0)),
        mode="edge",
    )
    return (
        (padded[:-2] + padded[1:-1] + padded[2:]) / 3.0
    ).astype(np.float32)

def windows_to_frame_probs(
    window_probs,
    window_starts,
    window_ends,
    total_frames,
    num_classes=NUM_CLASSES,
):
    p_sum = np.zeros((total_frames, num_classes), dtype=np.float64)
    count = np.zeros(total_frames, dtype=np.float64)

    for i, start in enumerate(window_starts):
        end = int(window_ends[i])
        p_sum[start:end + 1] += window_probs[i]
        count[start:end + 1] += 1.0

    uncovered = count == 0
    if np.any(uncovered):
        # Any truly uncovered frame defaults to D0X / No gesture.
        count[uncovered] = 1.0
        p_sum[uncovered, 0] = 1.0

    return (p_sum / count[:, None]).astype(np.float32)

def labels_to_segments(labels):
    labels = np.asarray(labels, dtype=np.int64)
    if len(labels) == 0:
        return []

    segments = []
    start = 0
    current = int(labels[0])

    for i in range(1, len(labels)):
        value = int(labels[i])
        if value != current:
            segments.append({
                "label": current,
                "start": start,
                "end": i - 1,
            })
            start = i
            current = value

    segments.append({
        "label": current,
        "start": start,
        "end": len(labels) - 1,
    })
    return segments

def gesture_segments(labels):
    return [
        seg for seg in labels_to_segments(labels)
        if int(seg["label"]) != 0
    ]

def link_same_label_across_gaps(
    frame_labels,
    max_gap_frames,
    fps,
):
    linked = np.asarray(frame_labels, dtype=np.int64).copy()
    log = []

    while True:
        segs = labels_to_segments(linked)
        candidates = []

        for i, left in enumerate(segs):
            label = int(left["label"])
            if label == 0:
                continue

            for j in range(i + 1, len(segs)):
                right = segs[j]
                gap = int(right["start"] - left["end"] - 1)

                if gap > max_gap_frames:
                    break

                if int(right["label"]) == label:
                    if gap > 0:
                        candidates.append((gap, int(left["start"]), i, j, label))
                    break

        if not candidates:
            break

        _, _, i, j, label = min(candidates, key=lambda x: (x[0], x[1]))

        # Rebuild segments because a previous pass may have modified them.
        segs = labels_to_segments(linked)
        left = segs[i]
        right = segs[j]

        gap_start = int(left["end"] + 1)
        gap_end = int(right["start"] - 1)
        old_labels = linked[gap_start:gap_end + 1].copy()
        gap_frames = gap_end - gap_start + 1

        linked[gap_start:gap_end + 1] = label

        log.append({
            "label": label,
            "gap_start": gap_start,
            "gap_end": gap_end,
            "gap_frames": gap_frames,
            "gap_sec": gap_frames / fps,
            "replaced_labels": sorted(np.unique(old_labels).astype(int).tolist()),
        })

    return linked, log

def remove_short_gesture_segments(
    frame_labels,
    min_frames,
    fps,
    fill_same_neighbors=True,
):
    src = np.asarray(frame_labels, dtype=np.int64)
    out = src.copy()
    segs = labels_to_segments(src)
    log = []

    for i, seg in enumerate(segs):
        label = int(seg["label"])
        if label == 0:
            continue

        n_frames = int(seg["end"] - seg["start"] + 1)
        if n_frames >= min_frames:
            continue

        new_label = 0
        reason = "short_to_background"

        if fill_same_neighbors and 0 < i < len(segs) - 1:
            prev_seg = segs[i - 1]
            next_seg = segs[i + 1]
            prev_len = int(prev_seg["end"] - prev_seg["start"] + 1)
            next_len = int(next_seg["end"] - next_seg["start"] + 1)

            if (
                int(prev_seg["label"]) != 0
                and int(prev_seg["label"]) == int(next_seg["label"])
                and prev_len >= min_frames
                and next_len >= min_frames
            ):
                new_label = int(prev_seg["label"])
                reason = "short_between_same_label"

        out[int(seg["start"]):int(seg["end"]) + 1] = new_label

        log.append({
            "start_frame": int(seg["start"]),
            "end_frame": int(seg["end"]),
            "duration_frames": n_frames,
            "duration_sec": n_frames / fps,
            "old_label": label,
            "new_label": new_label,
            "reason": reason,
        })

    return out, log

def temporal_iou(a, b):
    inter = max(
        min(a["end"], b["end"]) - max(a["start"], b["start"]) + 1,
        0,
    )
    union = max(a["end"], b["end"]) - min(a["start"], b["start"]) + 1
    return 0.0 if union <= 0 else inter / union

def overlap_score(gt, pred, theta_frames):
    used = set()
    scores = []

    gt = sorted(gt, key=lambda x: x["start"])
    pred = sorted(pred, key=lambda x: x["start"])

    for g in gt:
        candidates = []

        for j, p in enumerate(pred):
            if j in used or p["label"] != g["label"]:
                continue
            if abs(p["start"] - g["start"]) > theta_frames:
                continue
            if abs(p["end"] - g["end"]) > theta_frames:
                continue

            candidates.append((temporal_iou(p, g), j))

        if candidates:
            score, j = max(candidates, key=lambda x: x[0])
            used.add(j)
            scores.append(score)
        else:
            scores.append(0.0)

    unmatched_pred = len(pred) - len(used)
    denom = len(gt) + unmatched_pred
    return 1.0 if denom == 0 else float(np.sum(scores) / denom)

def build_ground_truth(selected_rows, total_frames):
    gt_frame_labels = np.zeros(total_frames, dtype=np.int64)
    gt_segments = []

    for row in selected_rows.itertuples(index=False):
        label = int(row.class_idx)
        start = max(0, int(row.t_start) - 1)
        end = min(total_frames - 1, int(row.t_end) - 1)

        if end < start:
            continue

        gt_frame_labels[start:end + 1] = label

        if label != 0:
            gt_segments.append({
                "label": label,
                "start": start,
                "end": end,
            })

    return gt_frame_labels, gt_segments

def evaluate_continuous_video(
    model,
    video_path,
    selected_rows,
    window_size=WINDOW_SIZE,
    window_step=WINDOW_STEP,
    model_frames=MODEL_FRAMES,
    enable_smoothing=ENABLE_SMOOTHING,
    t_avg_sec=T_AVG_SEC,
    min_gesture_sec=MIN_GESTURE_SEC,
    fill_same_neighbors=FILL_SAME_NEIGHBORS,
    theta_sec=THETA_SEC,
    show_progress=False,
):
    total_frames, fps = get_video_info(video_path)
    starts = build_window_starts(total_frames, window_size, window_step)
    ends = [
        min(start + window_size - 1, total_frames - 1)
        for start in starts
    ]
    sample_offsets = build_sample_offsets(window_size, model_frames)

    p_raw = np.zeros((len(starts), NUM_CLASSES), dtype=np.float32)

    t0 = time.perf_counter()

    iterator = enumerate(starts)
    if show_progress:
        iterator = tqdm(
            iterator,
            total=len(starts),
            desc=Path(video_path).name,
        )

    for wi, start in iterator:
        frames_np, _ = read_sampled_window(
            video_path,
            start,
            total_frames,
            sample_offsets,
        )
        _, _, probs = run_inference(model, frames_np, DEVICE)
        p_raw[wi] = probs

    infer_elapsed = time.perf_counter() - t0

    p_smooth = smooth_window_probs(p_raw, enable_smoothing)

    p_raw_frame = windows_to_frame_probs(
        p_raw, starts, ends, total_frames
    )
    p_smooth_frame = windows_to_frame_probs(
        p_smooth, starts, ends, total_frames
    )

    raw_labels = np.argmax(p_raw_frame, axis=1)
    smooth_labels = np.argmax(p_smooth_frame, axis=1)

    link_gap_frames = int(round(t_avg_sec * fps))
    min_gesture_frames = int(round(min_gesture_sec * fps))
    theta_frames = int(round(theta_sec * fps))

    linked_labels, linking_log = link_same_label_across_gaps(
        smooth_labels,
        link_gap_frames,
        fps,
    )

    final_labels, removal_log = remove_short_gesture_segments(
        linked_labels,
        min_gesture_frames,
        fps,
        fill_same_neighbors=fill_same_neighbors,
    )

    gt_labels, gt_segments = build_ground_truth(
        selected_rows,
        total_frames,
    )

    stages = {
        "Raw": raw_labels,
        "After smoothing": smooth_labels,
        "After linking": linked_labels,
        "After removing (final)": final_labels,
    }

    metric_rows = []
    for stage_name, labels in stages.items():
        pred_segments = gesture_segments(labels)
        metric_rows.append({
            "Stage": stage_name,
            "Frame-wise Accuracy": float(np.mean(labels == gt_labels)),
            "Temporal Overlap Score": overlap_score(
                gt_segments,
                pred_segments,
                theta_frames,
            ),
            "Gesture segments": len(pred_segments),
        })

    metrics = pd.DataFrame(metric_rows)

    final_segments = gesture_segments(final_labels)

    return {
        "video_path": Path(video_path),
        "total_frames": total_frames,
        "fps": fps,
        "duration_sec": total_frames / fps,
        "window_size": window_size,
        "window_step": window_step,
        "overlap_ratio": 1.0 - window_step / window_size,
        "window_starts": starts,
        "window_ends": ends,
        "num_windows": len(starts),
        "sample_offsets": sample_offsets,
        "P_raw": p_raw,
        "P_smooth": p_smooth,
        "P_raw_frame": p_raw_frame,
        "P_smooth_frame": p_smooth_frame,
        "raw_labels": raw_labels,
        "smooth_labels": smooth_labels,
        "linked_labels": linked_labels,
        "final_labels": final_labels,
        "gt_labels": gt_labels,
        "gt_segments": gt_segments,
        "final_segments": final_segments,
        "linking_log": linking_log,
        "removal_log": removal_log,
        "metrics": metrics,
        "infer_elapsed_sec": infer_elapsed,
        "mean_infer_sec_per_window": (
            infer_elapsed / len(starts) if starts else np.nan
        ),
    }

def final_segment_table(result):
    rows = []
    fps = result["fps"]

    for i, seg in enumerate(result["final_segments"], start=1):
        label = int(seg["label"])
        start = int(seg["start"])
        end = int(seg["end"])

        rows.append({
            "segment_id": i,
            "class_idx": label,
            "class_code": CLASS_CODES[label],
            "gesture": CLASS_DESCRIPTIONS[label],
            "start_frame": start,
            "end_frame": end,
            "start_sec": start / fps,
            "end_sec": end / fps,
            "duration_sec": (end - start + 1) / fps,
        })

    return pd.DataFrame(rows)

def plot_continuous_timeline(result):
    labels = [
        ("Ground truth", result["gt_labels"]),
        ("Raw", result["raw_labels"]),
        ("Smoothed", result["smooth_labels"]),
        ("Linked", result["linked_labels"]),
        ("Final", result["final_labels"]),
    ]

    fig, axes = plt.subplots(
        len(labels),
        1,
        figsize=(16, 10),
        sharex=True,
    )

    x = np.arange(result["total_frames"])
    for ax, (name, y) in zip(axes, labels):
        ax.plot(x, y)
        ax.set_ylabel(name)
        ax.grid(alpha=0.2)

    axes[-1].set_xlabel("Frame")
    fig.suptitle(result["video_path"].name)
    plt.tight_layout()
    plt.show()


### 5.2 Full continuous test split

This loop reuses the exact same `evaluate_continuous_video()` function as the one-video test. No global state is mutated or restored between videos.


In [ ]:
ALL_MAX_VIDEOS = None     # set e.g. 5 for a quick test
ALL_SAVE_CSV = True
SAVE_RAW_POSTPROCESS_CACHE = True
POSTPROCESS_CACHE_PATH = "/kaggle/working/videomamba_postprocess_raw_cache.pkl"

valid_test = df_test[df_test["video_found"]].copy()
valid_test["_video_key"] = valid_test["video"].apply(canonical_video_key)

video_groups = []
for video_key, rows in valid_test.groupby("_video_key", sort=True):
    rows = rows.sort_values("t_start").copy()
    path = Path(rows.iloc[0]["video_path"])
    if path.is_file():
        video_groups.append((video_key, path, rows))

if ALL_MAX_VIDEOS is not None:
    video_groups = video_groups[:int(ALL_MAX_VIDEOS)]

records = []

# Raw inference cache: one entry per video. This is reused by Section 5.3,
# so post-processing ablations do NOT rerun VideoMamba forward inference.
continuous_raw_cache = {}

total_raw_correct = 0
total_final_correct = 0
total_frames_weighted = 0

for video_key, path, rows in tqdm(
    video_groups,
    total=len(video_groups),
    desc="Continuous test videos",
):
    result = evaluate_continuous_video(
        model,
        path,
        rows,
        show_progress=False,
    )

    # Keep only the raw model output + metadata required to rerun
    # smoothing/linking/removing and metrics without model inference.
    continuous_raw_cache[path.name] = {
        "video": path.name,
        "fps": float(result["fps"]),
        "total_frames": int(result["total_frames"]),
        "window_starts": np.asarray(result["window_starts"], dtype=np.int64),
        "window_ends": np.asarray(result["window_ends"], dtype=np.int64),
        "P_raw": np.asarray(result["P_raw"], dtype=np.float32),
        "gt_labels": np.asarray(result["gt_labels"], dtype=np.int64),
        "gt_segments": [dict(seg) for seg in result["gt_segments"]],
    }

    metric_map = result["metrics"].set_index("Stage")

    raw_acc = float(metric_map.loc["Raw", "Frame-wise Accuracy"])
    raw_overlap = float(metric_map.loc["Raw", "Temporal Overlap Score"])
    final_acc = float(
        metric_map.loc["After removing (final)", "Frame-wise Accuracy"]
    )
    final_overlap = float(
        metric_map.loc["After removing (final)", "Temporal Overlap Score"]
    )

    total_frames = int(result["total_frames"])
    total_raw_correct += int(np.sum(result["raw_labels"] == result["gt_labels"]))
    total_final_correct += int(np.sum(result["final_labels"] == result["gt_labels"]))
    total_frames_weighted += total_frames

    records.append({
        "video": path.name,
        "frames": total_frames,
        "fps": result["fps"],
        "duration_sec": result["duration_sec"],
        "windows": result["num_windows"],
        "overlap_percent": 100.0 * result["overlap_ratio"],
        "raw_frame_accuracy": raw_acc,
        "raw_temporal_overlap": raw_overlap,
        "final_frame_accuracy": final_acc,
        "final_temporal_overlap": final_overlap,
        "final_gesture_segments": len(result["final_segments"]),
        "link_operations": len(result["linking_log"]),
        "removal_changes": len(result["removal_log"]),
        "inference_sec": result["infer_elapsed_sec"],
        "mean_ms_per_window": 1000.0 * result["mean_infer_sec_per_window"],
    })

continuous_per_video = pd.DataFrame(records)

dataset_summary = pd.DataFrame({
    "Metric": [
        "Videos",
        "Mean per-video raw Frame Accuracy",
        "Mean per-video final Frame Accuracy",
        "Global raw Frame Accuracy",
        "Global final Frame Accuracy",
        "Mean per-video raw Temporal Overlap",
        "Mean per-video final Temporal Overlap",
    ],
    "Value": [
        len(continuous_per_video),
        continuous_per_video["raw_frame_accuracy"].mean(),
        continuous_per_video["final_frame_accuracy"].mean(),
        total_raw_correct / total_frames_weighted,
        total_final_correct / total_frames_weighted,
        continuous_per_video["raw_temporal_overlap"].mean(),
        continuous_per_video["final_temporal_overlap"].mean(),
    ],
})

display(dataset_summary)
display(continuous_per_video.head())

if ALL_SAVE_CSV:
    CONTINUOUS_CSV = "/kaggle/working/videomamba_ipn_continuous_per_video.csv"
    CONTINUOUS_SUMMARY_CSV = "/kaggle/working/videomamba_ipn_continuous_summary.csv"

    continuous_per_video.to_csv(CONTINUOUS_CSV, index=False)
    dataset_summary.to_csv(CONTINUOUS_SUMMARY_CSV, index=False)

    print("Saved:")
    print(" -", CONTINUOUS_CSV)
    print(" -", CONTINUOUS_SUMMARY_CSV)


# ---------------------------------------------------------------------
# Save the raw inference cache so Section 5.3 can be rerun independently
# of VideoMamba inference within the same Kaggle session.
# ---------------------------------------------------------------------
if SAVE_RAW_POSTPROCESS_CACHE:
    import pickle
    with open(POSTPROCESS_CACHE_PATH, "wb") as f:
        pickle.dump(continuous_raw_cache, f, protocol=pickle.HIGHEST_PROTOCOL)

    cache_bytes = Path(POSTPROCESS_CACHE_PATH).stat().st_size
    print(
        f"Raw post-processing cache saved: {POSTPROCESS_CACHE_PATH} "
        f"({cache_bytes / 1024**2:.2f} MiB)"
    )
    print(
        "Post-processing ablations can now be rerun without model inference."
    )


### 5.4 P1: Segment-level CVPR metrics and error/control example manifests

RAW window prediction only. Reports F1@10/25/50 temporal IoU, Edit, IPN diagnostic Levenshtein, ECE/Brier/NLL, plus correctly and incorrectly recognized segments. Manifests are for later visualization. No external GPU pass.


In [ ]:
# P1 offline RAW continuous evaluation + manifest of wrong AND correct examples.
# Segmental F1@{10,25,50} and Edit follow CVPR MS-TCN conventions.
# These scores are NOT RSC's causal Online-F1/Early@10/TTC.
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

P1_IPN_DIR = Path("/kaggle/working/p1_ipn_re_evaluation")
P1_IPN_DIR.mkdir(parents=True, exist_ok=True)

def p1_segments(labels, ignore_background=True):
    labels = np.asarray(labels, dtype=np.int64)
    if not len(labels):
        return []
    boundaries = np.r_[0, np.flatnonzero(labels[1:] != labels[:-1])+1, len(labels)]
    return [(int(labels[a]), int(a), int(b-1)) for a, b in zip(boundaries[:-1], boundaries[1:])
            if not (ignore_background and int(labels[a]) == 0)]

def p1_edit_distance(a, b):
    prev = list(range(len(b)+1))
    for i, x in enumerate(a, 1):
        curr = [i]+[0]*len(b)
        for j, y in enumerate(b, 1):
            curr[j] = min(curr[j-1]+1, prev[j]+1, prev[j-1]+(x!=y))
        prev = curr
    return int(prev[-1])

def p1_segment_counts(gt_segs, pred_segs, threshold):
    # One-to-one class-consistent temporal IoU matching.
    matched = set()
    tp, fp = 0, 0
    for p_label, p_start, p_end in pred_segs:
        best_i, best_iou = None, -1.0
        for i, (g_label, g_start, g_end) in enumerate(gt_segs):
            if i in matched or p_label != g_label:
                continue
            intersection = max(0, min(p_end, g_end)-max(p_start, g_start)+1)
            union = (p_end-p_start+1)+(g_end-g_start+1)-intersection
            iou = intersection / union if union else 0.0
            if iou > best_iou:
                best_i, best_iou = i, iou
        if best_i is not None and best_iou >= threshold:
            matched.add(best_i)
            tp += 1
        else:
            fp += 1
    return tp, fp, len(gt_segs)-tp

def p1_ece(probs, y, bins=15):
    conf = np.max(probs, axis=1)
    correct = (np.argmax(probs, axis=1) == y).astype(float)
    edges = np.linspace(0.0, 1.0, bins+1)
    ece = 0.0
    for k in range(bins):
        in_bin = (conf >= edges[k]) & ((conf < edges[k+1]) if k < bins-1 else (conf <= edges[k+1]))
        if np.any(in_bin):
            ece += np.mean(in_bin)*abs(np.mean(correct[in_bin])-np.mean(conf[in_bin]))
    return float(ece)

# Isolated calibration, with unchanged raw softmax (no test-set tuning).
iso = isolated_results.copy()
prob_cols = [f"prob_{i}" for i in range(NUM_CLASSES)]
assert set(prob_cols).issubset(iso.columns), "Run updated isolated Section 4 first."
probs = iso[prob_cols].to_numpy(dtype=float)
y_iso = iso["gt_idx"].to_numpy(dtype=int)
assert np.allclose(probs.sum(axis=1), 1., atol=1e-4)
brier = float(np.mean(np.sum((probs-np.eye(NUM_CLASSES)[y_iso])**2, axis=1)))
calibration = {"n": int(len(y_iso)), "ece_15": p1_ece(probs, y_iso),
               "multiclass_brier_sum": brier, "nll": float(
                   -np.log(np.clip(probs[np.arange(len(y_iso)), y_iso], 1e-12, 1)).mean())}
(P1_IPN_DIR/"isolated_calibration.json").write_text(json.dumps(calibration, indent=2))
iso_wrong = iso[iso.gt_idx != iso.pred_idx].copy().sort_values("confidence", ascending=False)
iso_right = iso[iso.gt_idx == iso.pred_idx].copy().sort_values("confidence", ascending=False)
iso_wrong.to_csv(P1_IPN_DIR/"isolated_error_cases.csv", index=False)
iso_right.to_csv(P1_IPN_DIR/"isolated_correct_cases.csv", index=False)

assert continuous_raw_cache, "Run full continuous Section 5.2 first."
per_video = []
all_gt, all_pred = [], []
counts = {t: np.zeros(3, dtype=int) for t in (0.10, 0.25, 0.50)}
errors, controls = [], []
for name, entry in sorted(continuous_raw_cache.items()):
    gt = np.asarray(entry["gt_labels"], dtype=np.int64)
    p_frame = windows_to_frame_probs(
        np.asarray(entry["P_raw"]), np.asarray(entry["window_starts"]),
        np.asarray(entry["window_ends"]), len(gt))
    pred = p_frame.argmax(axis=1).astype(np.int64)
    assert len(gt) == len(pred) and len(gt)>0
    gs, ps = p1_segments(gt), p1_segments(pred)
    row = {"video": name, "frames": len(gt),
           "frame_accuracy": float(accuracy_score(gt, pred)),
           "macro_f1": float(f1_score(gt, pred, labels=list(range(NUM_CLASSES)),
                                        average="macro", zero_division=0))}
    for t in counts:
        a = np.array(p1_segment_counts(gs, ps, t), dtype=int)
        counts[t] += a
        row[f"segment_f1_at_{int(t*100)}"] = (2*a[0]/(2*a[0]+a[1]+a[2])) if (2*a[0]+a[1]+a[2]) else 0.
    dist = p1_edit_distance([s[0] for s in gs], [s[0] for s in ps])
    row["edit_distance"] = dist
    row["edit_score_ms_tcn_pct"] = 100*(1-dist/max(len(gs), len(ps), 1))
    row["ipn_gt_normalized_levenshtein_pct"] = 100*(1-dist/max(len(gs),1))
    per_video.append(row)
    all_gt.append(gt); all_pred.append(pred)
    for correct, sink in ((False, errors), (True, controls)):
        match = (gt == pred) if correct else (gt != pred)
        # For controls require a gesture; avoid selecting easy background-only frames.
        if correct:
            match &= (gt != 0)
        for _, a, b in p1_segments(match.astype(np.int64), ignore_background=True):
            mid = (a+b)//2
            sink.append({"video": name, "video_path": str(resolve_video_path(name)),
                         "start": a+1, "end": b+1, "indexing": "IPN_1_based",
                         "gt_idx": int(gt[mid]), "pred_idx": int(pred[mid]),
                         "duration_frames": b-a+1, "correct": bool(correct)})
per = pd.DataFrame(per_video)
per.to_csv(P1_IPN_DIR/"continuous_raw_segment_metrics_per_video.csv", index=False)
for filename, cases in (("continuous_error_cases.csv", errors), ("continuous_correct_cases.csv", controls)):
    df = pd.DataFrame(cases).sort_values("duration_frames", ascending=False) if cases else pd.DataFrame()
    df.to_csv(P1_IPN_DIR/filename, index=False)
gt_all = np.concatenate(all_gt); pr_all = np.concatenate(all_pred)
summary = {"videos": len(per), "frames": len(gt_all),
           "global_frame_accuracy": float(accuracy_score(gt_all, pr_all)),
           "global_macro_f1": float(f1_score(gt_all, pr_all, labels=list(range(NUM_CLASSES)),
                                              average="macro", zero_division=0)),
           "edit_score_ms_tcn_mean_pct": float(per.edit_score_ms_tcn_pct.mean()),
           "ipn_levenshtein_gt_normalized_mean_pct_diagnostic": float(per.ipn_gt_normalized_levenshtein_pct.mean()),
           "protocol": "raw offline overlapping windows; IPN gesture labels exclude class 0 in segment metrics"}
for t, (tp, fp, fn) in counts.items():
    summary[f"segment_f1_at_{int(t*100)}"] = float(2*tp/(2*tp+fp+fn)) if (2*tp+fp+fn) else 0.
(P1_IPN_DIR/"continuous_raw_segment_metrics_summary.json").write_text(json.dumps(summary, indent=2))
print("Isolated calibration:", calibration)
print("Continuous RAW summary:", summary)
print("Visualization case CSVs exported; no image style is imposed.")


### Additional continuous event metrics: P/R@10/25/50, matched boundary MAE and false positives per minute

Uses existing RAW window cache; no inference. Boundary errors computed only on class-matched segments with temporal IoU>=0.50. Not causal latency.


In [ ]:
# P1 additional CONTINUOUS diagnostics: TP/FP/FN, precision/recall at
# tIoU=0.10/0.25/0.50, strict matched onset/offset MAE (seconds),
# false positives per video minute and over-segmentation ratio.
# Matched boundary MAE is conditioned on one-to-one correct-class tIoU>=0.50.
from pathlib import Path
import json, numpy as np, pandas as pd
P1_EXTRA_CONT_DIR = P1_IPN_DIR / "extra_continuous_metrics"
P1_EXTRA_CONT_DIR.mkdir(parents=True,exist_ok=True)
P1_COUNTS = p1_segment_counts
P1_THRESHOLDS=(.10,.25,.50)
p1_counts_global={th:np.zeros(3,dtype=int) for th in P1_THRESHOLDS}
p1_class_counts={k:np.zeros(3,dtype=int) for k in range(1,NUM_CLASSES)}
p1_total_seconds=0.;p1_gt_count=0;p1_pred_count=0
boundary_rows=[];video_rows=[]

def p1_match_boundary(gt,pred,thresh=.50):
    pairs=sorted(((p1_tiou(g,p),gi,pi)
                  for gi,g in enumerate(gt) for pi,p in enumerate(pred)
                  if g[0]==p[0]),reverse=True)
    used_g=set();used_p=set();matches=[]
    for iou,gi,pi in pairs:
        if iou<thresh: break
        if gi not in used_g and pi not in used_p:
            matches.append((gi,pi,float(iou)))
            used_g.add(gi);used_p.add(pi)
    return matches

# Independent local tIoU helper; metrics cell does not require visualization cell.
def p1_tiou(a,b):
    inter=max(0,min(a[2],b[2])-max(a[1],b[1])+1)
    union=(a[2]-a[1]+1)+(b[2]-b[1]+1)-inter
    return inter/union if union else 0.

for vid,cache in sorted(continuous_raw_cache.items()):
    gt=np.asarray(cache["gt_labels"],dtype=np.int64)
    p_frame=windows_to_frame_probs(
        np.asarray(cache["P_raw"]),np.asarray(cache["window_starts"]),
        np.asarray(cache["window_ends"]),len(gt))
    pred=p_frame.argmax(axis=1).astype(np.int64)
    fps=max(float(cache["fps"]),1e-6)
    gs,ps=p1_segments(gt),p1_segments(pred)
    p1_total_seconds+=len(gt)/fps
    p1_gt_count+=len(gs);p1_pred_count+=len(ps)
    row={"video":str(vid),"gt_segments":len(gs),"pred_segments":len(ps),
         "duration_seconds":len(gt)/fps}
    for th in P1_THRESHOLDS:
        cnt=np.asarray(P1_COUNTS(gs,ps,th),dtype=int)
        p1_counts_global[th]+=cnt
        tp,fp,fn=map(int,cnt)
        row[f"precision_at_{int(th*100)}"]=tp/(tp+fp) if (tp+fp) else 0.
        row[f"recall_at_{int(th*100)}"]=tp/(tp+fn) if (tp+fn) else 0.
        row[f"tp_at_{int(th*100)}"]=tp
        row[f"fp_at_{int(th*100)}"]=fp
        row[f"fn_at_{int(th*100)}"]=fn
        row[f"f1_at_{int(th*100)}"]=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0.
    for label in range(1,NUM_CLASSES):
        c=np.asarray(P1_COUNTS(
            [x for x in gs if x[0]==label],
            [x for x in ps if x[0]==label],.50),dtype=int)
        p1_class_counts[label]+=c
    for gi,pi,iou in p1_match_boundary(gs,ps):
        g,p=gs[gi],ps[pi]
        boundary_rows.append({
            "video":str(vid),"gt_class":int(g[0]),"gt_start":g[1],"gt_end":g[2],
            "pred_start":p[1],"pred_end":p[2],"tiou":iou,
            "onset_signed_error_sec":(p[1]-g[1])/fps,
            "offset_signed_error_sec":(p[2]-g[2])/fps,
            "onset_absolute_error_sec":abs(p[1]-g[1])/fps,
            "offset_absolute_error_sec":abs(p[2]-g[2])/fps
        })
    video_rows.append(row)
pd.DataFrame(video_rows).to_csv(
    P1_EXTRA_CONT_DIR/"per_video_detection_precision_recall.csv",index=False)
pd.DataFrame(boundary_rows).to_csv(
    P1_EXTRA_CONT_DIR/"matched_onset_offset_errors.csv",index=False)
pc=[]
for label,counts in p1_class_counts.items():
    tp,fp,fn=map(int,counts)
    pc.append({"class_idx":label,"class_name":CLASS_CODES[label],
               "tp":tp,"fp":fp,"fn":fn,
               "precision_at_50":tp/(tp+fp) if tp+fp else 0.,
               "recall_at_50":tp/(tp+fn) if tp+fn else 0.,
               "f1_at_50":2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.})
pd.DataFrame(pc).to_csv(
    P1_EXTRA_CONT_DIR/"per_class_segment_f1_at_50.csv",index=False)
report={"videos":len(video_rows),"total_duration_minutes":p1_total_seconds/60.,
        "gt_gesture_segments":p1_gt_count,
        "pred_gesture_segments":p1_pred_count,
        "pred_to_gt_segment_ratio":p1_pred_count/max(p1_gt_count,1),
        "background_label_excluded_from_segments":True,
        "prediction_protocol":"RAW overlapping windows; OFFLINE",
        "boundary_protocol":"one-to-one correct-class tIoU>=0.50, conditioned on matched true positives"}
for th,counts in p1_counts_global.items():
    tp,fp,fn=map(int,counts)
    key=str(int(th*100))
    report[f"tp_at_{key}"]=tp; report[f"fp_at_{key}"]=fp;report[f"fn_at_{key}"]=fn
    report[f"precision_at_{key}"]=tp/(tp+fp) if tp+fp else 0.
    report[f"recall_at_{key}"]=tp/(tp+fn) if tp+fn else 0.
    report[f"f1_at_{key}"]=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
tp,fp,fn=map(int,p1_counts_global[.50])
report["unmatched_predicted_segments_per_minute_at_50"]=fp/(p1_total_seconds/60) if p1_total_seconds else None
report["miss_rate_at_50"]=fn/(tp+fn) if tp+fn else None
for field in ("onset_absolute_error_sec","offset_absolute_error_sec","onset_signed_error_sec","offset_signed_error_sec"):
    report["mean_"+field] = (float(pd.DataFrame(boundary_rows)[field].mean())
                          if boundary_rows else None)
report["matched_segment_count_at_50"]=len(boundary_rows)
(P1_EXTRA_CONT_DIR/"detection_boundary_summary.json").write_text(
    json.dumps(report,indent=2))
print("Continuous segment metrics + boundary diagnostics:");print(json.dumps(report,indent=2))


### P1 Continuous full-export: EXACT sixteen frames of a contributing window + GT/Pred timeline

Match GT and predicted gesture segments one-to-one at temporal IoU≥0.50. Export all correct and error events, including unmatched predictions; select a representative model window with maximum event overlap, record all contributing windows, and show frame-accurate GT vs offline RAW predictions. No extra inference.


In [ ]:
# P1 CONTINUOUS: exhaustively classify GT/predicted gesture segments with
# one-to-one class-aware tIoU>=0.50 matching; export 16 EXACT frames of a
# representative MODEL WINDOW + contextual GT/prediction timeline for each case.
# All other overlapping windows remain explicitly recorded as contributors.
# These figures are OFFLINE; never interpret framewise averaging as causal onset.
from pathlib import Path
import cv2, numpy as np, pandas as pd, json
from IPython.display import display, Image as IPyImage

P1_CONT_VIS_DIR = P1_IPN_DIR / "continuous_16frame_timeline"
P1_CONT_VIS_DIR.mkdir(parents=True, exist_ok=True)
P1_CONT_EXPORT_ALL = True
P1_CONT_LIMIT = None             # None = export all GT/pred events
P1_CONT_TILE = 192
P1_CONT_JPEG_QUALITY = 87
P1_CONT_PREVIEW = 2
P1_CONTEXT_SECONDS = 1.25
for group in ("correct", "error"):
    (P1_CONT_VIS_DIR/group).mkdir(exist_ok=True)

def p1_tiou(a,b):
    inter = max(0, min(a[2],b[2])-max(a[1],b[1])+1)
    union = (a[2]-a[1]+1)+(b[2]-b[1]+1)-inter
    return inter/union if union else 0.

def p1_match_05(gt_segments, pred_segments):
    # Global greedy, one-to-one, highest-class-consistent tIoU first.
    candidates = sorted(
        ((p1_tiou(g,p),gi,pi) for gi,g in enumerate(gt_segments)
         for pi,p in enumerate(pred_segments) if g[0]==p[0]),
        reverse=True,
    )
    matched_g, matched_p, matched = set(), set(), []
    for iou,gi,pi in candidates:
        if iou < .5:
            break
        if gi not in matched_g and pi not in matched_p:
            matched_g.add(gi); matched_p.add(pi)
            matched.append((gi,pi,float(iou)))
    return matched_g,matched_p,matched

def p1_choose_model_window(cache, seg):
    starts = np.asarray(cache["window_starts"],dtype=int)
    ends = np.asarray(cache["window_ends"],dtype=int)
    overlap = np.maximum(0,np.minimum(ends,seg[2])-np.maximum(starts,seg[1])+1)
    center = (seg[1]+seg[2])/2
    distances = np.abs((starts+ends)/2-center)
    wi = int(np.lexsort((distances,-overlap))[0])
    # Model input follows the exact original window helper.
    offsets = build_sample_offsets(WINDOW_SIZE, MODEL_FRAMES)
    sampled = np.clip(starts[wi]+offsets,0,int(cache["total_frames"])-1).astype(int)
    # A frame's aggregated probability comes from all windows overlapping that frame.
    midpoint = int((seg[1]+seg[2])//2)
    contributing_to_mid = np.flatnonzero((starts<=midpoint)&(ends>=midpoint))
    contributing_to_event = np.flatnonzero(overlap>0)
    return wi, sampled, contributing_to_mid.tolist(), contributing_to_event.tolist()

def p1_render_continuous_case(case, gt, pred, cache, video_path, output):
    wi, source_indices, contributors_mid, contributors_event = p1_choose_model_window(
        cache,(case["gt_idx"],case["start"],case["end"]))
    fps = max(float(cache["fps"]),1e-6)
    header = (f"CONTINUOUS {case['status'].upper()} | {case['kind']} | "
              f"GT {CLASS_CODES[int(case['gt_idx'])]} "
              f"PRED {CLASS_CODES[int(case['pred_idx'])]} | tIoU {case['tiou']:.3f}")
    footer = (f"video {Path(str(video_path)).name} | event frames "
              f"{case['start']}..{case['end']} | model window #{wi} "
              f"= {int(cache['window_starts'][wi])}..{int(cache['window_ends'][wi])}"
              f" | contributors(mid) {len(contributors_mid)}")
    # Saved helper: original temporal indices AND spatial preprocessing.
    temp_mosaic = P1_CONT_VIS_DIR / "_temporary_mosaic.jpg"
    p1_mosaic_16(video_path, source_indices,header,footer,temp_mosaic)
    mosaic = cv2.imread(str(temp_mosaic))
    try:
        temp_mosaic.unlink(missing_ok=True)
    except Exception:
        pass
    if mosaic is None:
        raise RuntimeError("Unable to open intermediate 16-frame mosaic")
    width = mosaic.shape[1]
    context = max(8,int(round(P1_CONTEXT_SECONDS*fps)))
    left = max(0,int(case["start"])-context)
    right = min(len(gt)-1,int(case["end"])+context)
    # Use nearest-neighbor pixel mapping to avoid inventing/interpolating labels.
    indices = np.rint(np.linspace(left,right,width)).astype(int)
    colors = [tuple(int(x) for x in cv2.applyColorMap(
        np.array([[int(255*j/max(NUM_CLASSES-1,1))]],dtype=np.uint8),
        cv2.COLORMAP_TURBO)[0,0]) for j in range(NUM_CLASSES)]
    timeline = np.full((150,width,3),245,dtype=np.uint8)
    for offset,vals in ((45,gt),(94,pred)):
        for x,k in enumerate(indices):
            timeline[offset:offset+24,x] = colors[int(vals[k])]
    cv2.putText(timeline, "GT", (10,39), cv2.FONT_HERSHEY_SIMPLEX,.53,(0,0,0),1,cv2.LINE_AA)
    cv2.putText(timeline, "RAW Prediction", (10,88),
                cv2.FONT_HERSHEY_SIMPLEX,.53,(0,0,0),1,cv2.LINE_AA)
    x0 = int(round((int(case["start"])-left)/max(right-left,1)*(width-1)))
    x1 = int(round((int(case["end"])-left)/max(right-left,1)*(width-1)))
    for x in (x0,x1):
        cv2.line(timeline,(x,39),(x,122),(30,30,30),1,cv2.LINE_AA)
    cv2.putText(timeline,f"Context {left/fps:.2f}..{right/fps:.2f}s  |  "
                f"highlight event {int(case['start'])/fps:.2f}..{int(case['end'])/fps:.2f}s",
                (10,142),cv2.FONT_HERSHEY_SIMPLEX,.48,(20,20,20),1,cv2.LINE_AA)
    final = np.vstack((mosaic,timeline))
    if not cv2.imwrite(str(output),final,[cv2.IMWRITE_JPEG_QUALITY,P1_CONT_JPEG_QUALITY]):
        raise RuntimeError(f"Unable to write {output}")
    return {
        "representative_window_index":wi,
        "window_start":int(cache["window_starts"][wi]),
        "window_end":int(cache["window_ends"][wi]),
        "model_input_indices_zero_based":",".join(map(str,source_indices)),
        "contributing_windows_at_midpoint":",".join(map(str,contributors_mid)),
        "contributing_windows_over_event":",".join(map(str,contributors_event)),
        "number_of_contributing_windows_at_midpoint":len(contributors_mid),
        "number_of_contributing_windows_over_event":len(contributors_event),
    }

p1_cont_cases,p1_cont_runtime={},{}
for vkey,cache in sorted(continuous_raw_cache.items()):
    vid = str(vkey)
    gt = np.asarray(cache["gt_labels"],dtype=np.int64)
    p_frame = windows_to_frame_probs(cache["P_raw"],cache["window_starts"],
                                     cache["window_ends"],len(gt))
    pred = np.asarray(p_frame.argmax(axis=1),dtype=np.int64)
    gs,ps = p1_segments(gt),p1_segments(pred)
    used_g,used_p,matches = p1_match_05(gs,ps)
    cases=[]
    for gi,seg in enumerate(gs):
        if gi in used_g:
            _,pi,iou = next(m for m in matches if m[0]==gi)
            status,kind = "correct","matched_gt_gesture_tiou_ge_050"
            pred_idx=int(ps[pi][0])
        else:
            overlap = [p for p in ps if p1_tiou(seg,p)>0.]
            if any(p[0]==seg[0] for p in overlap):
                kind="boundary_or_fragmentation"
            elif overlap:
                kind="gesture_class_confusion"
            else:
                kind="missed_gesture"
            status="error"
            pred_idx=int(pred[(seg[1]+seg[2])//2])
            iou=max((p1_tiou(seg,p) for p in ps if p[0]==seg[0]),default=0.)
        cases.append({"video":vid,"status":status,"kind":kind,"source":"ground_truth",
                      "start":int(seg[1]),"end":int(seg[2]),"gt_idx":int(seg[0]),
                      "pred_idx":pred_idx,"tiou":float(iou)})
    for pi,seg in enumerate(ps):
        if pi in used_p:
            continue
        overlap = [g for g in gs if p1_tiou(seg,g)>0.]
        kind=("duplicate_or_fragmented_prediction" if any(g[0]==seg[0] for g in overlap)
              else "class_confusion_prediction" if overlap else "false_positive_gesture")
        cases.append({"video":vid,"status":"error","kind":kind,
                      "source":"unmatched_prediction","start":int(seg[1]),
                      "end":int(seg[2]),"gt_idx":int(gt[(seg[1]+seg[2])//2]),
                      "pred_idx":int(seg[0]),"tiou":float(max(
                          (p1_tiou(seg,g) for g in gs if g[0]==seg[0]),default=0.))})
    p1_cont_cases[vid]=cases
    video_path = (resolve_video_path(str(vid)))
    p1_cont_runtime[vid]=(gt,pred,cache,str(video_path))

all_cases=[x for subset in p1_cont_cases.values() for x in subset]
all_cases = sorted(all_cases,key=lambda x:(x["video"],x["start"],x["source"]))
if P1_CONT_LIMIT is not None:
    all_cases = all_cases[:int(P1_CONT_LIMIT)]
rows,failures=[],[]
for idx,case in enumerate(all_cases,1):
    gt,pred,cache,path=p1_cont_runtime[case["video"]]
    out=P1_CONT_VIS_DIR / case["status"] / f"{case['status']}_{idx:05d}.jpg"
    try:
        extra=p1_render_continuous_case(case,gt,pred,cache,path,out)
        rows.append({**case,**extra,"video_path":path,"image_path":str(out),
                     "duration_sec":(case["end"]-case["start"]+1)/float(cache["fps"]),
                     "sampled_frames":16,"protocol":"RAW offline overlapping-window aggregation"})
    except Exception as exc:
        failures.append({**case,"error":str(exc)})
pd.DataFrame(rows).to_csv(P1_CONT_VIS_DIR/"continuous_all_events_index.csv",index=False)
pd.DataFrame(failures).to_csv(P1_CONT_VIS_DIR/"continuous_export_failures.csv",index=False)
print(f"CONTINUOUS: exported {len(rows)}/{len(all_cases)} "
      f"cases (correct={sum(x['status']=='correct' for x in rows)}, "
      f"error={sum(x['status']=='error' for x in rows)}); "
      f"decode failures={len(failures)}. Out: {P1_CONT_VIS_DIR}")
for group in ("error","correct"):
    for image_path in sorted((P1_CONT_VIS_DIR/group).glob("*.jpg"))[:P1_CONT_PREVIEW]:
        display(IPyImage(filename=str(image_path),width=690))
